In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CARPETA = "/content/drive/MyDrive/fresas"
R = "/content/recortes"
M = "/content/maduracion_recortes"
!unzip -q -o "$CARPETA/recortes.zip" -d /content
!unzip -q -o "$CARPETA/maduracion_recortes.zip" -d /content
!cp "$CARPETA/etiquetas_propias.csv" "$CARPETA/clasificador.pt" "$CARPETA/clasificador.json" /content/

In [ ]:
import random
import numpy as np
import pandas as pd

eti = pd.read_csv("/content/etiquetas_propias.csv")
pro = eti[eti.etiqueta.isin(["sana", "danada"])].assign(fuente="propias", ruta=lambda d: R + "/caja/" + d.recorte)
pro["parte"] = np.where(pro.grupo.isin(["escena02", "escena07", "escena11"]), "validacion", "entrenamiento")

mad = pd.read_csv(f"{M}/recortes.csv").rename(columns={"clase": "etiqueta", "foto": "grupo"})
mad = mad.assign(fuente="maduracion", ruta=lambda d: M + "/" + d.etiqueta + "/" + d.recorte)
fotos_mad = sorted(mad.grupo.unique())
random.Random(42).shuffle(fotos_mad)
mad["parte"] = np.where(mad.grupo.isin(fotos_mad[:len(fotos_mad) // 4]), "validacion", "entrenamiento")

rec = pd.read_csv(f"{R}/recortes.csv")
pub = rec[rec.fuente != "propias"].assign(etiqueta=lambda d: d.clase, ruta=lambda d: R + "/caja/" + d.recorte)
rng = random.Random(42)
parte = {}
for _, g in pub.groupby(["fuente", "clase"]):
    grupos = sorted(g.grupo.unique())
    rng.shuffle(grupos)
    n = max(1, round(len(grupos) * 0.15))
    for i, gr in enumerate(grupos):
        parte[gr] = "prueba" if i < n else "validacion" if i < 2 * n else "entrenamiento"
pub["parte"] = pub.grupo.map(parte)
pub = pub[pub.parte == "entrenamiento"].groupby(["fuente", "etiqueta"]).sample(150, random_state=0)

columnas = ["ruta", "fuente", "grupo", "etiqueta", "parte"]
datos = pd.concat([pro[columnas], mad[columnas], pub[columnas]], ignore_index=True)
datos["y"] = (datos.etiqueta == "danada").astype(int)
assert datos.groupby("grupo").parte.nunique().max() == 1
print(datos.groupby(["parte", "fuente", "etiqueta"]).size().unstack(fill_value=0))

In [ ]:
import json, shutil, time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image

disp = "cuda" if torch.cuda.is_available() else "cpu"
cfg = json.load(open("/content/clasificador.json"))
TAM, MEDIA, DESV = cfg["tamano"], cfg["media"], cfg["desviacion"]
sin_aumento = transforms.Compose([
    transforms.Resize((TAM, TAM)), transforms.ToTensor(), transforms.Normalize(MEDIA, DESV)])
con_aumento = transforms.Compose([
    transforms.RandomResizedCrop(TAM, scale=(0.7, 1), ratio=(0.8, 1.25)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(20),
    transforms.ColorJitter(0.25, 0.25, 0.15, 0.02),
    transforms.ToTensor(), transforms.Normalize(MEDIA, DESV)])


class Recortes(Dataset):
    def __init__(self, tabla, trans):
        self.rutas, self.y, self.trans = tabla.ruta.tolist(), tabla.y.tolist(), trans

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.trans(Image.open(self.rutas[i]).convert("RGB")), self.y[i]


def predecir(m, tabla):
    m.eval()
    ps = []
    with torch.no_grad():
        for x, _ in DataLoader(Recortes(tabla, sin_aumento), batch_size=32, num_workers=2):
            ps.append(torch.softmax(m(x.to(disp)), 1)[:, 1].cpu())
    return torch.cat(ps).numpy()


def f1_danadas(y, p, umbral=0.5):
    pred = p >= umbral
    vp, fp, fn = (pred & (y == 1)).sum(), (pred & (y == 0)).sum(), (~pred & (y == 1)).sum()
    return 2 * vp / (2 * vp + fp + fn) if vp else 0.0


assert f1_danadas(np.array([1, 1, 0, 0]), np.array([0.9, 0.2, 0.7, 0.1])) == 0.5
print("Listo en", disp)

In [ ]:
EPOCAS = 5
ent, val = datos[datos.parte == "entrenamiento"], datos[datos.parte == "validacion"]
cuenta = ent.groupby(["fuente", "etiqueta"]).ruta.transform("count")
muestreo = WeightedRandomSampler(torch.tensor((1 / cuenta).values, dtype=torch.double), num_samples=1200, replacement=True)
c_ent = DataLoader(Recortes(ent, con_aumento), batch_size=32, sampler=muestreo, num_workers=2)

m = models.efficientnet_b0()
m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
m.load_state_dict(torch.load("/content/clasificador.pt", map_location=disp))
m = m.to(disp)
torch.manual_seed(0)
opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=1e-4)
perdida = nn.CrossEntropyLoss()

p = predecir(m, val)
mejor = f1_danadas(val.y.values, p)
print("Antes de ajustar, F1 de dañadas en validación:", round(mejor, 3))
torch.save(m.state_dict(), "/content/ajustado.pt")
for ep in range(EPOCAS):
    t0 = time.time()
    m.train()
    for x, y in c_ent:
        opt.zero_grad()
        perdida(m(x.to(disp)), y.to(disp)).backward()
        opt.step()
    p = predecir(m, val)
    f1 = f1_danadas(val.y.values, p)
    print(f"época {ep + 1}: F1 de dañadas en validación {f1:.3f} ({time.time() - t0:.0f} s)")
    if f1 > mejor:
        mejor = f1
        torch.save(m.state_dict(), "/content/ajustado.pt")

m.load_state_dict(torch.load("/content/ajustado.pt", map_location=disp))
p = predecir(m, val)
for fuente in ["propias", "maduracion"]:
    s = (val.fuente == fuente).values
    y = val.y.values[s]
    print(f"Validación {fuente}: dañadas encontradas {((p[s] >= 0.5) & (y == 1)).sum()} de {y.sum()}, "
          f"sanas marcadas como dañadas {((p[s] >= 0.5) & (y == 0)).sum()} de {(y == 0).sum()}")

shutil.copy(f"{CARPETA}/clasificador.pt", f"{CARPETA}/clasificador_base.pt")
shutil.copy("/content/ajustado.pt", f"{CARPETA}/clasificador.pt")
cfg["experimento"] = cfg["experimento"] + "_ajustado"
json.dump(cfg, open(f"{CARPETA}/clasificador.json", "w"), indent=1)
print("Guardado en Drive:", CARPETA + "/clasificador.pt")